# Resume Screening & Skill Matching — End-to-End Demo

This notebook runs the entire project **straight from GitHub**, so you can verify it works without any local setup. Just choose **Runtime → Run all**.

**What this notebook demonstrates**
1. Clone the repo + install dependencies
2. **Core matching** — rank résumés against a job description (SBERT semantic + skill overlap)
3. **Explainability** — matched/missing skills with evidence + upskilling roadmap
4. **Multilingual** — an English JD matching Spanish/French résumés cross-lingually
5. **Active learning** — recruiter feedback → trained re-ranker → adaptive scoring (with interpretable coefficients)
6. **Learning curve** — metrics improving as feedback accumulates
7. **Benchmark metrics** — Precision / Recall / F1 / MRR / NDCG

Everything runs on CPU. The first model load downloads a ~470 MB multilingual model, which takes about a minute.

## 1. Setup — clone the repo and install dependencies

In [ ]:
# Clone the public repository (no login needed)
!git clone https://github.com/devn-cmd/Resume-Matcher.git
%cd Resume-Matcher

In [ ]:
# Install the runtime dependencies (fast path).
# For the exact pinned versions instead, use:  !pip install -r requirements.txt
!pip install -q sentence-transformers spacy langdetect scikit-learn pandas pdfplumber python-docx matplotlib

In [ ]:
# Make the repo importable and confirm the data shipped with the clone
import sys
sys.path.insert(0, '.')
print("Eval data:")
!ls data/eval
print("\nDemo résumés:")
!ls data/raw/resumes

## 2. Core matching — rank résumés against a job description\n\nLoads the sample job description and the 10 demo résumés, then ranks them by fit.

In [ ]:
from pathlib import Path
import pandas as pd
from src.matching import ResumeMatcher
from src.ingestion import read_document

# Sample job description (Data Scientist)
jd = Path('data/raw/jd_data_scientist.txt').read_text(encoding='utf-8')

# Read every résumé in data/raw/resumes (PDF / DOCX / TXT)
resume_dir = Path('data/raw/resumes')
resumes = {f.name: read_document(str(f)) for f in sorted(resume_dir.iterdir()) if f.is_file()}

matcher = ResumeMatcher()
results = matcher.rank(jd, resumes)

pd.DataFrame([{
    'Candidate':      r.resume_id,
    'Language':       r.language,
    'Cross-lingual':  r.cross_lingual,
    'Final score':    r.final_score,
    'Semantic':       r.semantic_score,
    'Skill match':    r.skill_score,
    'Suitable':       matcher.is_suitable(r),
} for r in results])

## 3. Explainability — why did the top candidate match?\n\nShows the matched skills with the exact résumé sentence as evidence, plus the personalized upskilling roadmap for any missing skills.

In [ ]:
import config
from src.skills import SkillExtractor
from src.explain import explain

extractor = SkillExtractor(config.SKILLS_DB_PATH)
top = results[0]
info = explain(resumes[top.resume_id], set(top.matched_skills), set(top.missing_skills), extractor)

print(f"TOP CANDIDATE: {top.resume_id}\n")
print("Matched skills (with evidence):")
for skill, sentence in info['evidence'].items():
    print(f"  • {skill}: {sentence}")

print(f"\nMissing skills: {sorted(top.missing_skills) or '—'}")

if info['recommendations']:
    print("\nUpskilling roadmap for missing skills:")
    for rec in info['recommendations']:
        print(f"  • {rec['skill']} (market demand {rec['demand_score']})")
        for step in rec['roadmap']:
            print(f"      Step {step['step']}: {step['action']}")

## 4. Multilingual — cross-lingual matching\n\nThe English job description is matched directly against résumés written in other languages, with no translation step.

In [ ]:
cross = [r for r in results if r.cross_lingual]
if cross:
    print("Résumés matched cross-lingually (different language from the JD):\n")
    for r in cross:
        print(f"  • {r.resume_id}  —  detected {r.language}, "
              f"final score {r.final_score:.2f}, suitable={matcher.is_suitable(r)}")
else:
    print("No cross-lingual résumés in this set.")

## 5. Active learning — the recruiter feedback loop

The system learns from recruiter shortlist/reject decisions. A small, **interpretable** logistic-regression layer (a classification layer) is trained on accumulated feedback and re-scores candidates. The coefficients *are* the explanation — you can read exactly what drives each decision.

In [ ]:
from src.feedback import load_feedback, count_feedback, class_balance
from src.active_learning import ActiveLearningRanker

pos, neg = class_balance()
print(f"Feedback collected: {count_feedback()}  ({pos} shortlists / {neg} rejects)\n")

# Train the adaptive re-ranker on the accumulated feedback
ranker = ActiveLearningRanker()
report = ranker.fit(load_feedback())
print(f"Re-ranker trained: {report.trained}  (on {report.n_samples} samples)\n")

print("Learned coefficients (interpretability — positive pushes toward 'suitable'):")
for feature, weight in ranker.coefficients().items():
    print(f"  {feature:>18}: {weight:+.3f}")

Now re-rank the same candidates **with** the trained model. Compare the static blend against the adaptive score, and note which candidates the model flags as *uncertain* (its review queue).

In [ ]:
adaptive = matcher.rank_with_feedback(jd, resumes, ranker=ranker)

pd.DataFrame([{
    'Candidate':  r.resume_id,
    'Static':     r.final_score,
    'Adaptive':   round(r.adaptive_score, 4) if r.adaptive_score is not None else None,
    'In review queue': r.is_uncertain,
    'Suitable':   matcher.is_suitable(r),
} for r in adaptive])

## 6. Learning curve — does feedback actually improve the model?\n\nFirst, load the evaluation set, then show the saved learning curve (instant). The curve plots metrics on a held-out test slice against the number of recruiter labels collected.

In [ ]:
from evaluation.eval_data import load_eval_data
jd_texts, eval_resumes, labels = load_eval_data()

In [ ]:
from evaluation.active_learning_eval import plot_curve
from IPython.display import Image

curve = pd.read_csv('evaluation/active_learning_curve.csv')
display(curve)
plot_curve(curve)
Image('evaluation/active_learning_curve.png')

**Optional — regenerate the curve from scratch** (takes a few minutes on CPU).

⚠️ This resets the feedback log as part of the simulation. If you want the seeded feedback back afterward, re-run `!git checkout data/recruiter_feedback.jsonl`.

In [ ]:
from evaluation.active_learning_eval import run_active_learning_eval
fresh_curve = run_active_learning_eval(jd_texts, eval_resumes, labels,
                                       rounds=12, queries_per_round=5)
fresh_curve

## 7. Benchmark metrics\n\nThe standard evaluation: Precision, Recall, F1, MRR, NDCG, cosine separation, and timing on the full labelled set.

In [ ]:
# (If you ran the optional regen above, restore the seed feedback first.)
!git checkout data/recruiter_feedback.jsonl 2>/dev/null

from evaluation.evaluate import evaluate
metrics = evaluate(jd_texts, eval_resumes, labels)
metrics

## 8. (Optional) Launch the interactive dashboard

The Streamlit dashboard is best seen in the video walkthrough, but you can launch it here through a public tunnel if you want to click around. This step can be flaky and is **not required** to verify the project.

When the tunnel URL appears, open it; if it asks for a password, paste the IP printed just above the link.

In [ ]:
# OPTIONAL — interactive dashboard via localtunnel
!pip install -q streamlit
!streamlit run dashboard/app.py &>./streamlit_log.txt &
import time; time.sleep(8)
import urllib.request
print("Tunnel password (your Colab public IP):",
      urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode())
!npx --yes localtunnel --port 8501